# Phase 5 – Databricks + Olist Starter Notebook

In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window

base_path = "/Volumes/filestore/filestore/olist/"

### STEP 1: LOAD DATA

In [0]:
customers = spark.read.option("header", "true").csv(base_path + "olist_customers_dataset.csv")
orders = spark.read.option("header", "true").csv(base_path + "olist_orders_dataset.csv")
order_items = spark.read.option("header", "true").csv(base_path + "olist_order_items_dataset.csv")
products = spark.read.option("header", "true").csv(base_path + "olist_products_dataset.csv")
translation = spark.read.option("header", "true").csv(base_path + "product_category_name_translation.csv")

### STEP 2: CLEAN DATA

In [0]:
customers = customers.dropna(subset=["customer_id"])
orders = orders.dropna(subset=["order_id", "customer_id"])
order_items = order_items.dropna(subset=["order_id", "product_id"])
#Fixing Datatype
order_items = order_items.withColumn("price", col("price").cast("double"))

### STEP 3: FIX PRODUCT CATEGORY 

In [0]:
products = products.join(
    translation,
    products.product_category_name == translation.product_category_name,
    "left"
).select(
    "product_id",
    col("product_category_name_english").alias("product_category_name")
)

products = products.fillna({"product_category_name": "Unknown"})

### STEP 4: JOIN DATA (FACT + DIMENSION)

In [0]:
df = orders \
    .join(customers, "customer_id") \
    .join(order_items, "order_id") \
    .join(products, "product_id", "left")

# Tasks 

###Task 1: Top 3 Customers per City

In [0]:
customer_spend = df.groupBy("customer_id", "customer_city") \
                   .agg(sum("price").alias("total_spend"))

window_city = Window.partitionBy("customer_city") \
                    .orderBy(col("total_spend").desc())

top_customers = customer_spend.withColumn("rank", rank().over(window_city)) \
                             .filter(col("rank") <= 3)

display(top_customers.select(
    col("customer_city").alias("city"),
    "customer_id",
    "total_spend",
    "rank"
))

city,customer_id,total_spend,rank
abadia dos dourados,9e01f714a2b3b8962c222cf2b74c20dc,199.0,1
abadia dos dourados,a23e3f9a2b656b23b7e52075964b42cd,120.0,2
abadia dos dourados,f11eb8f0b8b87510a93e3e1aa10b0ade,39.9,3
abadiania,576d71ddb21b21763cfedce73b902180,949.99,1
abaete,d47c8bb51df6f716e196ecd6cd5d2c09,449.0,1
abaete,ff0d62f8be4c098e6306f39bc6ebded4,225.9,2
abaete,5371894984937a27cf40c7d20699a786,208.9,3
abaetetuba,c7eb06383ae604616cb4c9d36fe6745e,1500.0,1
abaetetuba,367fd22f1e994de87cf447d29c167a1f,797.6,2
abaetetuba,7727e2cc9ec428ad3173cc812ce1781b,580.27,3


### Task 2: Running Total of Sales

In [0]:
daily_sales = df.groupBy(to_date("order_purchase_timestamp").alias("date")) \
                .agg(sum("price").alias("daily_sales"))

window_date = Window.orderBy("date")

running_total = daily_sales.withColumn(
    "running_total",
    sum("daily_sales").over(window_date)
)

display(running_total)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


date,daily_sales,running_total
2016-09-04,72.89,72.89
2016-09-05,59.5,132.39
2016-09-15,134.97,267.36
2016-10-02,100.0,367.36
2016-10-03,463.48,830.84
2016-10-04,9940.96,10771.8
2016-10-05,8343.25,19115.05
2016-10-06,7960.51,27075.559999999998
2016-10-07,7228.05,34303.61
2016-10-08,8441.849999999999,42745.46


### TASK 3: Top Products per Category

In [0]:
product_sales = df.groupBy("product_category_name", "product_id") \
    .agg(sum("price").alias("total_sales"))

window_category = Window.partitionBy("product_category_name") \
    .orderBy(col("total_sales").desc())

top_products = product_sales.withColumn(
    "rank",
    dense_rank().over(window_category)
)

task3_result = top_products.select(
    col("product_category_name").alias("category"),
    "product_id",
    "total_sales",
    "rank"
).orderBy(
    when(col("category") == "Unknown", 1).otherwise(0),
    col("category"),
    col("rank")
)

display(task3_result)


category,product_id,total_sales,rank
agro_industry_and_commerce,11250b0d4b709fee92441c5f34122aed,9111.0,1
agro_industry_and_commerce,423a6644f0aa529e8828ff1f91003690,8043.0,2
agro_industry_and_commerce,672e757f331900b9deea127a2a7b79fd,6885.0,3
agro_industry_and_commerce,c183fd5d2abf05873fa6e1014ed9e06c,5934.6,4
agro_industry_and_commerce,2b69866f22de8dad69c976771daba91c,2990.0,5
agro_industry_and_commerce,c89226b8a795ae3d6bca9d90b20dbf04,2821.5,6
agro_industry_and_commerce,5fb0955cb683eb6f65a1f613e502eef5,2720.0,7
agro_industry_and_commerce,b7a60a397d4efd05c1b5d398fb9f9097,2399.0,8
agro_industry_and_commerce,cd5df6a3db7a3d064a55afd08289d762,2360.0,9
agro_industry_and_commerce,cd2f5c10e4e8dbc701f0bb68a09fdfe8,2199.0,10


### TASK 4: Customer Lifetime Value

In [0]:
clv = df.groupBy("customer_id") \
    .agg(sum("price").alias("total_spend"))

display(clv)


customer_id,total_spend
4fd75fb5ef1f01c7585bf746092a1544,39.0
a51b373c427132a77b112ca10f502c4f,159.99
f8975f0842ae1e4670a410a02b1b5053,64.9
d42ee2c04270ba237933c8bdd5e1c761,170.0
4e7656e34357b93f14b40c6400ca3f6e,144.99
13a4684a5a46fcbafb42d8e30af15219,21.5
f06a94a401e52fb019c72f2e8bbf6a2f,35.9
a207382f0f563c1f430ef732ade9d9ff,131.0
ebfb3f2e532c37d5a959ded000c9f603,31.99
938698ae953596e231326da5ff220f68,29.9


### TASK 5: Customer Segmentation

In [0]:
segmented = clv.withColumn(
    "segment",
    when(col("total_spend") > 10000, "Gold")
    .when((col("total_spend") >= 5000) & (col("total_spend") <= 10000), "Silver")
    .otherwise("Bronze")
)

segment_count = segmented.groupBy("segment").count()

display(segmented)
display(segment_count)

customer_id,total_spend,segment
4fd75fb5ef1f01c7585bf746092a1544,39.0,Bronze
a51b373c427132a77b112ca10f502c4f,159.99,Bronze
f8975f0842ae1e4670a410a02b1b5053,64.9,Bronze
d42ee2c04270ba237933c8bdd5e1c761,170.0,Bronze
4e7656e34357b93f14b40c6400ca3f6e,144.99,Bronze
13a4684a5a46fcbafb42d8e30af15219,21.5,Bronze
f06a94a401e52fb019c72f2e8bbf6a2f,35.9,Bronze
a207382f0f563c1f430ef732ade9d9ff,131.0,Bronze
ebfb3f2e532c37d5a959ded000c9f603,31.99,Bronze
938698ae953596e231326da5ff220f68,29.9,Bronze


segment,count
Bronze,98660
Silver,5
Gold,1


### TASK 6: Final Reporting Table

In [0]:
segmented_clean = segmented.select("customer_id", "segment")

final_df = df.groupBy("customer_id", "customer_city") \
    .agg(
        sum("price").alias("total_spend"),
        countDistinct("order_id").alias("total_orders")
    )

final_df = final_df.join(segmented_clean, "customer_id")

display(final_df.select(
    "customer_id",
    col("customer_city").alias("city"),
    "total_spend",
    "segment",
    "total_orders"
))

customer_id,city,total_spend,segment,total_orders
0728f6d108255cb411a50fb646c3c7dc,itapetinga,35.9,Bronze,1
3bbe55ec205026d44f2a0ce1059f6e5a,recife,510.0,Bronze,1
56dfafb543cfa4c2edcc08b9b2c5f0c6,sao paulo,95.0,Bronze,1
ba30b0a551ddef6057b752431dc4abeb,luis eduardo magalhaes,98.97,Bronze,1
fbac16c6734df62b9a470f3a5c8d7128,antonio prado,58.9,Bronze,1
737d438cd631428265f911322d7e7f9d,belo horizonte,79.99,Bronze,1
f9717b3ce18c05540645923e43f64710,vitoria,49.9,Bronze,1
ced3fb170a9a5007a9bc3558d49c35b1,pilar,43.9,Bronze,1
e6ac83a21d39a2e035df2c63aec86318,santos,39.99,Bronze,1
549ef037d5530887209688c8bb62d4c2,teresopolis,150.0,Bronze,1
